In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

In [16]:
import datasets 
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')   
df = dataset['train'].to_pandas()
df.describe()

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [17]:
judaism = df.loc[df['target_religion_jewish'] == True, ['comment_id', 'text', 'hate_speech_score']].drop_duplicates(subset='comment_id')
len(judaism)

1874

In [18]:
judaism.to_csv('data/ucberkeley-dlab_target_jewish.csv', index=False)
print(judaism['hate_speech_score'].describe())

count    1874.000000
mean       -0.857556
std         2.006435
min        -7.940000
25%        -2.180000
50%        -0.680000
75%         0.517500
max         5.090000
Name: hate_speech_score, dtype: float64


### Pilot Codebook Labeling

In [5]:
from dotenv import load_dotenv
import os
import anthropic
import json
import csv
import re
import pandas as pd
from config import UNIVERSAL, INPUT, I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

In [8]:
# Sanity Check (Sonnet across 3 runs for intra-model reliability)

judaism = pd.read_csv("data/ucberkeley-dlab_target_jewish.csv")

# test ids and texts
test_ids = [29933, 39476, 40464, 985, 32861, 32448, 27527, 20045]
test_texts = {row['comment_id']: row['text'] for _, row in judaism[judaism['comment_id'].isin(test_ids)].iterrows()}

# prompt blocks (block_name, block_content, is_not)
blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

MODEL_NAME = "claude-sonnet-4-6"
N_RUNS = 3

# results[run][comment_id][block_name] = [(code_id, label), ...]
results = {r: {} for r in range(1, N_RUNS + 1)}

for run in range(1, N_RUNS + 1):
    for comment_id in test_ids:
        results[run][comment_id] = {}
        text = test_texts[comment_id]

        for block_name, block_content, is_not in blocks:
            is_or_is_not = "is NOT" if is_not else "IS"
            system_text = UNIVERSAL.format(ISorisNOT=is_or_is_not) + block_content
            user_text = INPUT.format(id=comment_id, text=text)

            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=1024,
                temperature=0,
                system=[
                    {
                        "type": "text",
                        "text": system_text,
                        "cache_control": {"type": "ephemeral"},
                    }
                ],
                messages=[{"role": "user", "content": user_text}],
            )

            raw = response.content[0].text.strip()

            try:
                clean = re.sub(r'```json|```', '', raw).strip()
                data = json.loads(clean)
                parsed = [tuple(pair) for pair in list(data.values())[0]]
                results[run][comment_id][block_name] = parsed
            except Exception as e:
                results[run][comment_id][block_name] = {"parse_error": str(e), "raw": raw}

# save raw json
with open("test/comparative_results.json", "w") as f:
    json.dump(results, f, indent=2)

# flatten to dataframe (comment_id, block, code_id, run, label)
rows = []
for run in range(1, N_RUNS + 1):
    for comment_id, blocks_dict in results[run].items():
        for block_name, labels in blocks_dict.items():
            if isinstance(labels, list):
                for item in labels:
                    if len(item) == 2:
                        code_id, label = item
                        rows.append([comment_id, block_name, code_id, run, label])
                    else:
                        rows.append([comment_id, block_name, "MALFORMED", run, str(item)])
            else:
                rows.append([comment_id, block_name, "PARSE_ERROR", run, str(labels)])

flat_df = pd.DataFrame(rows, columns=["comment_id", "block", "code_id", "run", "label"])
flat_df.to_csv("test/comparative_results_flat.csv", index=False)

print(f"Done. {len(flat_df)} rows saved to comparative_results_flat.csv and comparative_results.json")

Done. 3048 rows saved to comparative_results_flat.csv and comparative_results.json


In [9]:
# Direct comparison across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

pivot_df = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
).reset_index()

pivot_df.to_csv("test/comparative_pivot.csv", index=False)
print(pivot_df.to_string(index=False))

 comment_id  block                 code_id 1 2 3
        985    I_1                  D1HATE N N N
        985    I_1              D1MANIFEST N N N
        985    I_1            D1PERCEPTION A A A
        985    I_1       D2COLLECTIVEBLAME I I I
        985    I_1            D2CONSPIRACY N N N
        985    I_1          D2ISRAELTARGET N N N
        985    I_1            D2STEREOTYPE I I I
        985    I_2               E1RADICAL N N N
        985    I_2              E1VIOLENCE N N N
        985    I_2            E2ALLEGATION I I I
        985    I_2       E2COLLECTIVEPOWER N N N
        985    I_2            E2CONSPIRACY N N N
        985    I_2        E2CONTROLECONOMY N N N
        985    I_2            E2CONTROLGOV N N N
        985    I_2          E2CONTROLMEDIA N N N
        985    I_2          E2CONTROLOTHER N N N
        985    I_2        E2DEHUMANIZATION N N N
        985    I_2              E2DEMONIZE N N N
        985    I_2            E2STEREOTYPE I I I
        985    I_2  

In [10]:
# Intra-model agreement across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

def pct_agreement(series_list):
    """Given a list of label-series aligned by index, return % where all match."""
    combined = pd.concat(series_list, axis=1)
    combined.columns = range(len(series_list))
    all_match = combined.apply(lambda row: row.nunique() == 1, axis=1)
    return all_match.mean() * 100

wide = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
)

run_cols = [c for c in wide.columns]
agreement_pct = pct_agreement([wide[c] for c in run_cols])

print(f"Sonnet: {agreement_pct:.2f}% full agreement across {len(run_cols)} runs")

pd.DataFrame([{"model": "sonnet", "in_model_agreement_pct": round(agreement_pct, 2)}]).to_csv(
    "test/in_model_agreement.csv", index=False
)

Sonnet: 98.72% full agreement across 3 runs


In [14]:
import pandas as pd
from itertools import combinations

df = pd.read_csv("test/comparative_results_flat.csv")

adjacent_4 = {
    frozenset(["N", "A"]),
    frozenset(["A", "I"]),
    frozenset(["I", "E"]),
}

def classify_disagreements(df, label_col, adjacent_pairs):
    wide = df.pivot_table(
        index=["comment_id", "block", "code_id"],
        columns="run",
        values=label_col,
        aggfunc="first"
    ).reset_index()
    run_cols = [c for c in wide.columns if c not in ["comment_id", "block", "code_id"]]

    adjacent_count = 0
    cross_count = 0
    cross_examples = []

    for _, row in wide.iterrows():
        labels = list({row[c] for c in run_cols if pd.notna(row[c])})
        if len(labels) > 1:
            for l1, l2 in combinations(labels, 2):
                pair = frozenset([l1, l2])
                if pair in adjacent_pairs:
                    adjacent_count += 1
                else:
                    cross_count += 1
                    cross_examples.append({
                        "comment_id": row["comment_id"],
                        "block": row["block"],
                        "code_id": row["code_id"],
                        "labels_seen": str(sorted(labels)),
                    })

    return adjacent_count, cross_count, cross_examples

adj, cross, cross_ex = classify_disagreements(df, "label", adjacent_4)

print("=== Categorical disagreement structure ===")
print(f"\nadjacent-bucket disagreements: {adj}")
print(f"cross-bucket disagreements:    {cross}")
if cross_ex:
    print("\ncross-bucket examples:")
    print(pd.DataFrame(cross_ex).to_string(index=False))

=== Categorical disagreement structure ===

adjacent-bucket disagreements: 12
cross-bucket disagreements:    1

cross-bucket examples:
 comment_id block           code_id labels_seen
      20045   I_3 E4HOLOCAUSTINTENT  ['I', 'N']


## Full Label

In [16]:
# create batches

import os, json, re
import pandas as pd
import anthropic
from dotenv import load_dotenv
from config import UNIVERSAL, INPUT, I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

judaism = pd.read_csv("data/ucberkeley-dlab_target_jewish.csv")
MODEL_NAME = "claude-sonnet-4-6"

blocks = [
    ("I_1",    I_1,    False),
    ("I_2",    I_2,    False),
    ("I_3",    I_3,    False),
    ("I_4",    I_4,    False),
    ("I_5",    I_5,    False),
    ("I_NO_1", I_NO_1, True),
    ("N_1",    N_1,    False),
    ("N_2",    N_2,    False),
    ("N_NO_1", N_NO_1, True),
    ("J_1",    J_1,    False),
    ("J_2",    J_2,    False),
    ("J_3",    J_3,    False),
    ("J_NO_1", J_NO_1, True),
]

os.makedirs("batches", exist_ok=True)
index_path = "batches/batch_index.json"

# load existing index so re-running this cell never double-submits
if os.path.exists(index_path):
    batch_index = json.load(open(index_path))
else:
    batch_index = {}

for block_name, block_content, is_not in blocks:
    if block_name in batch_index:
        print(f"{block_name}: already created (batch_id={batch_index[block_name]['batch_id']}), skipping")
        continue

    is_or_is_not = "is NOT" if is_not else "IS"
    system_text = UNIVERSAL.format(ISorisNOT=is_or_is_not) + block_content

    requests = []
    for _, row in judaism.iterrows():
        requests.append({
            "custom_id": str(row["comment_id"]),
            "params": {
                "model": MODEL_NAME,
                "max_tokens": 1024,
                "system": [{
                    "type": "text",
                    "text": system_text,
                    "cache_control": {"type": "ephemeral"},
                }],
                "messages": [{
                    "role": "user",
                    "content": INPUT.format(id=row["comment_id"], text=row["text"]),
                }],
            },
        })

    batch = client.messages.batches.create(requests=requests)
    batch_index[block_name] = {"batch_id": batch.id, "status": batch.processing_status}
    json.dump(batch_index, open(index_path, "w"), indent=2)
    print(f"{block_name}: created batch {batch.id}")

print("\nAll batches created. Index saved to batches/batch_index.json")

I_1: created batch msgbatch_01XLS6HXcoy85yEsQxF96Uyh
I_2: created batch msgbatch_0131pEP7eag4YKjVAiuqoPcC
I_3: created batch msgbatch_01YUcZbiyq5sffxyE6YtJKPA
I_4: created batch msgbatch_0127ynjwy5jikESR2JVayNu7
I_5: created batch msgbatch_0149b1FLRB9tjsgkjVYJAYfe
I_NO_1: created batch msgbatch_016FDWHb7dyz9PMgdDYfU5BP
N_1: created batch msgbatch_01CXrjeixxkGHHcMkHKJrQSi
N_2: created batch msgbatch_01MbRJoJUjoraC859gEKjpFu
N_NO_1: created batch msgbatch_01WitrAsR1huKqwhnNSNbmPA
J_1: created batch msgbatch_01BhEWhaUfBJBMYrgAwog4Go
J_2: created batch msgbatch_017FQbU1ZeCqiu2fKkswtN1S
J_3: created batch msgbatch_01Q7Hoew4R6CHHhFtF7GRb3v
J_NO_1: created batch msgbatch_01EWDWxDMY8SrqhUuDSkj4pz

All batches created. Index saved to batches/batch_index.json


In [24]:
# check and save batches
import json, re, os
import pandas as pd
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

index_path = "batches/batch_index.json"
batch_index = json.load(open(index_path))
os.makedirs("batches/results", exist_ok=True)

rows = []
print("=== Batch save status ===\n")

for block_name, info in batch_index.items():
    out_path = f"batches/results/{block_name}.csv"

    # already saved
    if os.path.exists(out_path):
        saved_df = pd.read_csv(out_path)
        print(f"  {block_name:10s}  already saved ({len(saved_df)} rows)")
        rows.append(pd.read_csv(out_path))
        continue

    batch = client.messages.batches.retrieve(info["batch_id"])
    status = batch.processing_status
    counts = batch.request_counts
    total = counts.processing + counts.succeeded + counts.errored + counts.canceled + counts.expired
    done  = counts.succeeded + counts.errored + counts.canceled + counts.expired

    if status != "ended":
        pct = (done / total * 100) if total > 0 else 0
        print(f"  {block_name:10s}  not done yet — {done}/{total} ({pct:.1f}%) [{status}]")
        continue

    # ended: parse and save
    block_rows = []
    for result in client.messages.batches.results(info["batch_id"]):
        comment_id = int(result.custom_id)
        if result.result.type != "succeeded":
            block_rows.append([comment_id, block_name, "API_ERROR", result.result.type])
            continue
        raw = result.result.message.content[0].text.strip()
        try:
            clean = re.sub(r'```json|```', '', raw).strip()
            data = json.loads(clean)
            for code_id, label in list(data.values())[0]:
                block_rows.append([comment_id, block_name, code_id, label])
        except Exception as e:
            block_rows.append([comment_id, block_name, "PARSE_ERROR", str(e)])

    df = pd.DataFrame(block_rows, columns=["comment_id", "block", "code_id", "label"])
    df.to_csv(out_path, index=False)
    batch_index[block_name]["status"] = "saved"
    print(f"  {block_name:10s}  saved {len(df)} rows -> {out_path}")
    rows.append(df)

json.dump(batch_index, open(index_path, "w"), indent=2)

# combine all saved blocks into a single flat file
if rows:
    combined = pd.concat(rows, ignore_index=True)
    combined.to_csv("batches/results/all_blocks_flat.csv", index=False)
    print(f"\nCombined flat file: {len(combined)} rows across {len(rows)} block(s) saved to batches/results/all_blocks_flat.csv")

=== Batch save status ===

  I_1         already saved (13070 rows)
  I_2         already saved (23942 rows)
  I_3         already saved (11229 rows)
  I_4         already saved (22290 rows)
  I_5         already saved (20604 rows)
  I_NO_1      already saved (1874 rows)
  N_1         already saved (7478 rows)
  N_2         already saved (33086 rows)
  N_NO_1      already saved (14964 rows)
  J_1         already saved (33409 rows)
  J_2         already saved (12902 rows)
  J_3         already saved (6259 rows)
  J_NO_1      already saved (10079 rows)

Combined flat file: 211186 rows across 13 block(s) saved to batches/results/all_blocks_flat.csv


#### Handle Missingness

#### Check Label Distribution of Full Dataset

In [30]:
import os
import pandas as pd

block_order = ["I_1", "I_2", "I_3", "I_4", "I_5", "I_NO_1", "N_1", "N_2", "N_NO_1", "J_1", "J_2", "J_3", "J_NO_1"]
error_types = ["API_ERROR", "PARSE_ERROR", "EMPTY_RESPONSE"]

dfs = []
for block_name in block_order:
    path = f"batches/results/{block_name}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        dfs.append(df[~df["code_id"].isin(error_types)])

combined = pd.concat(dfs, ignore_index=True)

total = len(combined)
counts = combined["label"].value_counts()
pcts = (counts / total * 100).round(2)

print(f"=== Label distribution across all successful rows (n={total:,}) ===\n")
for label in ["N", "E", "I", "A"]:
    n = counts.get(label, 0)
    p = pcts.get(label, 0)
    print(f"  {label}:  {n:>8,}  ({p:.2f}%)")

print(f"N is {'the most' if counts.get('N',0) == counts.max() else 'a'} frequent label so it makes a clean baseline.")

=== Label distribution across all successful rows (n=208,691) ===

  N:   196,220  (94.02%)
  E:     3,626  (1.74%)
  I:     5,758  (2.76%)
  A:     3,087  (1.48%)
N is the most frequent label so it makes a clean baseline.
